In [1]:
%%capture
!pip install transformers datasets

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

from tqdm.auto import tqdm
from transformers import BertTokenizer

This is a template of the notebook that you should complete and enrich with your own code.

First cells will be the same than the ones of the lab on text convolution.

# Data loading


In [3]:
dataset = load_dataset("stanfordnlp/imdb", split="train")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


# Pre-processing / Tokenization

In PyTorch, everything is tensor. Words are replaced by indices. A sentence, is therefore a sequence of indices (long integers). In the first HW, we constructed a `WhiteSpaceTokenizer`. Here we will use an already built tokenizer. It is more appropriate to transformers. It relies on sub-word units, and converts everything in lower case. This is not always the best choice, but here it will be sufficient. To quote the documentation, this tokenizer allows us to:

- Tokenize (splitting strings in sub-word token strings), converttokens strings to ids and back, and encoding/decoding (i.e., tokenizing and converting to integers).
- Add new tokens to the vocabulary in a way that is independent of the underlying structure (BPE, SentencePiece…).
- Manage special tokens (like mask, beginning-of-sentence, etc.): adding them, assigning them to attributes in the tokenizer for easy access and making sure they are not split during tokenization.

Here we are going to use the tokenizer from the well known Bert model, that we can directly download.

In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

In [5]:
def preprocessing_fn(x, tokenizer):
    x["input_ids"] = tokenizer(
        x["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=256,
        padding=False,
        return_attention_mask=False,
    )["input_ids"]
    return x

## 3.1 Data Preprocessing — Step 1: Load and tokenize the IMDB dataset

We shuffle the full dataset, select 10 000 samples, tokenize with the BERT tokenizer,
keep only `input_ids`, and split 80 % / 20 % into `document_train_set` / `document_valid_set`.
We do **not** use the sentiment labels because Word2Vec is an unsupervised model.


In [6]:
n_samples = 10000  # total number of examples used

# Shuffle the dataset so we get a representative mix of positive/negative reviews
dataset = dataset.shuffle(seed=42)

# Select n_samples samples
dataset = dataset.select(range(n_samples))

# Tokenize the dataset (batched=True for speed)
dataset = dataset.map(
    lambda x: preprocessing_fn(x, tokenizer),
    batched=True,
)

# Keep only the input_ids column — we don't need labels (unsupervised)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "input_ids"])

# 80 / 20 train / validation split
split = dataset.train_test_split(test_size=0.2, seed=42)

document_train_set = split["train"]
document_valid_set = split["test"]

print(f"Train documents : {len(document_train_set)}")
print(f"Valid documents : {len(document_valid_set)}")
print("Sample input_ids (first 10 tokens):", document_train_set[0]["input_ids"][:10])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train documents : 8000
Valid documents : 2000
Sample input_ids (first 10 tokens): [1045, 4669, 27368, 1999, 2395, 4959, 1010, 1998, 1045, 6618]


## 3.1 — Step 2: `extract_words_contexts`

For every position `i` in a document we extract the target word `w = doc[i]` and its
positive context `C⁺ = [doc[i-R], …, doc[i-1], doc[i+1], …, doc[i+R]]`.

**Border strategy**: we simply skip positions where the full window of radius `R` is not
available, i.e. we only keep indices `i` such that `R ≤ i < len(doc) - R`.  
This guarantees every `C⁺` has exactly `2R` entries without any padding or wrapping,
which keeps the implementation clean and avoids introducing artificial (padding-word,
real-word) pairs that would distort the embeddings.


In [7]:
def extract_words_contexts(doc_ids: list[int], R: int) -> tuple[list[int], list[list[int]]]:
    """
    Extract all (word, positive_context) pairs from a single tokenised document.

    Border strategy: skip the first and last R positions so every context
    window has exactly 2R tokens with no padding.

    Parameters
    ----------
    doc_ids : list of token ids representing one document
    R       : context radius

    Returns
    -------
    word_ids    : list of centre-word ids
    context_ids : list of lists, each of length 2R (positive context for the corresponding word)
    """
    word_ids = []
    context_ids = []

    for i in range(R, len(doc_ids) - R):
        centre = doc_ids[i]
        # Left context: positions [i-R, …, i-1]  +  Right context: positions [i+1, …, i+R]
        context = doc_ids[i - R : i] + doc_ids[i + 1 : i + R + 1]
        word_ids.append(centre)
        context_ids.append(context)

    return word_ids, context_ids

## 3.1 — Step 3: `flatten_dataset_to_list`

We'll apply `extract_words_contexts` over every document in the HuggingFace dataset and
concatenate the results into two flat lists.


In [8]:
def flatten_dataset_to_list(
    hf_dataset, R: int
) -> tuple[list[int], list[list[int]]]:
    """
    Apply extract_words_contexts on every document of a HuggingFace dataset
    and return two flat lists: all word ids and all positive context id lists.

    Parameters
    ----------
    hf_dataset : HuggingFace Dataset with an 'input_ids' column
    R          : context radius

    Returns
    -------
    all_word_ids    : flat list of centre-word ids
    all_context_ids : flat list of positive context lists (each of length 2R)
    """
    all_word_ids = []
    all_context_ids = []

    for sample in hf_dataset:
        doc_ids = sample["input_ids"]
        # Documents shorter than 2R+1 tokens have no valid centre word
        if len(doc_ids) < 2 * R + 1:
            continue
        word_ids, context_ids = extract_words_contexts(doc_ids, R)
        all_word_ids.extend(word_ids)
        all_context_ids.extend(context_ids)

    return all_word_ids, all_context_ids

## 3.1 — Step 4: Apply `flatten_dataset_to_list` to train and validation sets


In [9]:
# Hyperparameter: context radius
R = 5

print("Flattening training set …")
train_word_ids, train_context_ids = flatten_dataset_to_list(document_train_set, R)

print("Flattening validation set …")
valid_word_ids, valid_context_ids = flatten_dataset_to_list(document_valid_set, R)

print(f"\nR = {R}")
print(f"Train pairs  : {len(train_word_ids):,}")
print(f"Valid pairs  : {len(valid_word_ids):,}")
print(f"Context size : {len(train_context_ids[0])} (= 2R = {2 * R})")

Flattening training set …
Flattening validation set …

R = 5
Train pairs  : 1,573,630
Valid pairs  : 390,063
Context size : 10 (= 2R = 10)


## 3.1 — Step 5: PyTorch `Dataset` wrappers


In [10]:
class Word2VecDataset(Dataset):
    """
    Stores (word_id, positive_context_ids) pairs.
    Negative sampling is handled later in collate_fn so it can be re-sampled
    at every batch, giving more variety during training.
    """

    def __init__(self, word_ids: list[int], context_ids: list[list[int]]):
        assert len(word_ids) == len(context_ids)
        self.word_ids = word_ids
        self.context_ids = context_ids

    def __len__(self) -> int:
        return len(self.word_ids)

    def __getitem__(self, idx: int) -> tuple[int, list[int]]:
        return self.word_ids[idx], self.context_ids[idx]


train_set = Word2VecDataset(train_word_ids, train_context_ids)
valid_set = Word2VecDataset(valid_word_ids, valid_context_ids)

print(f"train_set length : {len(train_set):,}")
print(f"valid_set length : {len(valid_set):,}")

train_set length : 1,573,630
valid_set length : 390,063


## 3.1 — Step 6: `collate_fn` — adds negative context to each batch

For each batch, we randomly sample `2 * K * R` words from the whole vocabulary as
negative examples (one set per word in the batch).  
Sampling is done uniformly over `[0, vocab_size)`, which is the standard Word2Vec
approximation (the original paper uses a smoothed unigram distribution, but uniform
sampling is a common and acceptable simplification).


In [11]:
def make_collate_fn(K: int, vocab_size: int):
    """
    Factory that returns a collate function parametrised by K and vocab_size.

    Parameters
    ----------
    K          : negative-to-positive ratio (number of negatives = K * 2R per word)
    vocab_size : size of the vocabulary (used for random sampling)

    Returns
    -------
    collate_fn : function that turns a list of (word_id, pos_ctx) pairs into a dict
                 with keys 'word_id', 'positive_context_ids', 'negative_context_ids'.
    """

    def collate_fn(batch: list[tuple[int, list[int]]]) -> dict[str, torch.Tensor]:
        word_ids_list, pos_ctx_list = zip(*batch)
        batch_size = len(word_ids_list)
        context_size = len(pos_ctx_list[0])  # = 2R
        neg_size = K * context_size           # = 2KR negatives per word

        word_id_tensor = torch.tensor(word_ids_list, dtype=torch.long)           # (B,)
        pos_ctx_tensor = torch.tensor(pos_ctx_list, dtype=torch.long)            # (B, 2R)
        # Sample negatives uniformly from the vocabulary
        neg_ctx_tensor = torch.randint(                                           # (B, 2KR)
            low=0, high=vocab_size, size=(batch_size, neg_size), dtype=torch.long
        )

        return {
            "word_id": word_id_tensor,
            "positive_context_ids": pos_ctx_tensor,
            "negative_context_ids": neg_ctx_tensor,
        }

    return collate_fn

## 3.1 — Step 7: Wrap in `DataLoader`


In [12]:
# Hyperparameters
K = 5          # negative-sampling ratio
B = 512        # batch size

VOCAB_SIZE = tokenizer.vocab_size  # 30 522 for bert-base-uncased

collate_fn = make_collate_fn(K=K, vocab_size=VOCAB_SIZE)

train_loader = DataLoader(
    train_set,
    batch_size=B,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=False,
)

valid_loader = DataLoader(
    valid_set,
    batch_size=B,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
)

## 3.1 — Step 8: Sanity check — iterate 3 batches and print shapes


In [13]:
print(f"R = {R}   K = {K}   B = {B}   vocab_size = {VOCAB_SIZE}\n")

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 3:
        break
    print(f"--- Batch {batch_idx} ---")
    print(f"  word_id               shape : {batch['word_id'].shape}          (B,)")
    print(f"  positive_context_ids  shape : {batch['positive_context_ids'].shape}    (B, 2R={2*R})")
    print(f"  negative_context_ids  shape : {batch['negative_context_ids'].shape}  (B, 2KR={2*K*R})")
    print()

R = 5   K = 5   B = 512   vocab_size = 30522

--- Batch 0 ---
  word_id               shape : torch.Size([512])          (B,)
  positive_context_ids  shape : torch.Size([512, 10])    (B, 2R=10)
  negative_context_ids  shape : torch.Size([512, 50])  (B, 2KR=50)

--- Batch 1 ---
  word_id               shape : torch.Size([512])          (B,)
  positive_context_ids  shape : torch.Size([512, 10])    (B, 2R=10)
  negative_context_ids  shape : torch.Size([512, 50])  (B, 2KR=50)

--- Batch 2 ---
  word_id               shape : torch.Size([512])          (B,)
  positive_context_ids  shape : torch.Size([512, 10])    (B, 2R=10)
  negative_context_ids  shape : torch.Size([512, 50])  (B, 2KR=50)



# 3.2 Model

## 3.2 — Step 1: `Word2Vec` module

The model maintains two embedding tables:
- `Ew` (word embeddings): maps centre words to vectors `w ∈ ℝᵈ`.
- `Ec` (context embeddings): maps context words (positive and negative) to vectors `c ∈ ℝᵈ`.

Having two separate tables is standard in Word2Vec; it avoids the trivial solution
where a word would be most similar to itself.

The `forward` pass returns:
- `pos_scores` — sigmoid similarities for positive contexts, shape `(B, 2R)`.
- `neg_scores` — sigmoid similarities for negative contexts, shape `(B, 2KR)`.


In [14]:
class Word2Vec(nn.Module):
    """
    Word2Vec with negative sampling (SGNS).

    Parameters
    ----------
    vocab_size : number of tokens in the vocabulary
    d          : embedding dimension
    """

    def __init__(self, vocab_size: int, d: int):
        super().__init__()
        # Word (centre) embeddings — Ew in the assignment notation
        self.Ew = nn.Embedding(vocab_size, d)
        # Context embeddings — Ec in the assignment notation
        self.Ec = nn.Embedding(vocab_size, d)

        # Initialise with small uniform values for stable early training
        nn.init.uniform_(self.Ew.weight, -0.1, 0.1)
        nn.init.uniform_(self.Ec.weight, -0.1, 0.1)

    def forward(
        self,
        word_ids: torch.Tensor,           # (B,)
        positive_context_ids: torch.Tensor,  # (B, 2R)
        negative_context_ids: torch.Tensor,  # (B, 2KR)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Compute sigmoid similarity scores for positive and negative contexts.

        Returns
        -------
        pos_scores : (B, 2R)   — σ(c⊤ w) for positive context words
        neg_scores : (B, 2KR)  — σ(c⊤ w) for negative context words
        """
        # Centre-word embeddings: (B, d)
        w = self.Ew(word_ids)

        # Positive context embeddings: (B, 2R, d)
        c_pos = self.Ec(positive_context_ids)
        # Negative context embeddings: (B, 2KR, d)
        c_neg = self.Ec(negative_context_ids)

        # Dot products: (B, 2R) and (B, 2KR)
        # bmm needs (B, 1, d) x (B, d, 2R) → (B, 1, 2R) → squeeze
        pos_dot = torch.bmm(c_pos, w.unsqueeze(-1)).squeeze(-1)   # (B, 2R)
        neg_dot = torch.bmm(c_neg, w.unsqueeze(-1)).squeeze(-1)   # (B, 2KR)

        # Sigmoid similarity scores — σ(c⊤ w)
        pos_scores = torch.sigmoid(pos_dot)   # (B, 2R)
        neg_scores = torch.sigmoid(neg_dot)   # (B, 2KR)

        return pos_scores, neg_scores

## Loss function

From equation (1) in the assignment:
$$
\mathcal{L} = -\log \sigma(c^\top w) \text{ for } c \in C^+
            - \log(1 - \sigma(c^\top w)) \text{ for } c \in C^-
$$
This is binary cross-entropy: BCE with target 1 for positive pairs and target 0 for
negative pairs, averaged over all pairs in the batch.


In [15]:
def word2vec_loss(
    pos_scores: torch.Tensor,  # (B, 2R)
    neg_scores: torch.Tensor,  # (B, 2KR)
) -> torch.Tensor:
    """
    Binary cross-entropy loss for Word2Vec negative sampling.

    Positive pairs have target label 1; negative pairs have target label 0.
    The loss is averaged over all (word, context) pairs in the batch.
    """
    # -log σ(c⊤ w)   for positive pairs
    loss_pos = -torch.log(pos_scores + 1e-8).mean()
    # -log(1 − σ(c⊤ w)) for negative pairs
    loss_neg = -torch.log(1.0 - neg_scores + 1e-8).mean()
    return loss_pos + loss_neg

## 3.2 — Step 2 & 3: Training and validation


In [16]:
def train_epoch(
    model: Word2Vec,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """Run one training epoch and return the mean loss."""
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, leave=False, desc="train"):
        word_ids = batch["word_id"].to(device)
        pos_ctx = batch["positive_context_ids"].to(device)
        neg_ctx = batch["negative_context_ids"].to(device)

        optimiser.zero_grad()
        pos_scores, neg_scores = model(word_ids, pos_ctx, neg_ctx)
        loss = word2vec_loss(pos_scores, neg_scores)
        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: Word2Vec,
    loader: DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    """
    Evaluate mean loss and binary accuracy on the validation set.

    Accuracy: positive scores predicted > 0.5  AND negative scores < 0.5.
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, leave=False, desc="eval"):
        word_ids = batch["word_id"].to(device)
        pos_ctx = batch["positive_context_ids"].to(device)
        neg_ctx = batch["negative_context_ids"].to(device)

        pos_scores, neg_scores = model(word_ids, pos_ctx, neg_ctx)
        loss = word2vec_loss(pos_scores, neg_scores)
        total_loss += loss.item()

        # Accuracy: fraction of scores on the right side of 0.5
        correct += (pos_scores > 0.5).sum().item()
        correct += (neg_scores < 0.5).sum().item()
        total += pos_scores.numel() + neg_scores.numel()

    return total_loss / len(loader), correct / total

In [17]:
# Hyperparameters
d = 64   # embedding dimension
E = 5    # number of training epochs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Word2Vec(vocab_size=VOCAB_SIZE, d=d).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses = []
valid_losses = []
valid_accs = []

for epoch in range(1, E + 1):
    train_loss = train_epoch(model, train_loader, optimiser, device)
    valid_loss, valid_acc = evaluate(model, valid_loader, device)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    valid_accs.append(valid_acc)

    print(
        f"Epoch {epoch}/{E}  "
        f"train_loss={train_loss:.4f}  "
        f"valid_loss={valid_loss:.4f}  "
        f"valid_acc={valid_acc:.4f}"
    )

Using device: cpu


train:   0%|          | 0/3074 [00:00<?, ?it/s]

eval:   0%|          | 0/762 [00:00<?, ?it/s]

Epoch 1/5  train_loss=0.5621  valid_loss=0.4901  valid_acc=0.9099


train:   0%|          | 0/3074 [00:00<?, ?it/s]

...

eval:   0%|          | 0/762 [00:00<?, ?it/s]

Epoch 4/5  train_loss=0.4429  valid_loss=0.4833  valid_acc=0.9119


train:   0%|          | 0/3074 [00:00<?, ?it/s]

eval:   0%|          | 0/762 [00:00<?, ?it/s]

Epoch 5/5  train_loss=0.4346  valid_loss=0.4851  valid_acc=0.9119


## 3.2 — Step 4: `save_model`

Saves the word-embedding weights (`Ew`) to a file whose name encodes all
hyperparameters, making it easy to compare checkpoints later.


In [18]:
def save_model(
    model: Word2Vec,
    d: int,
    R: int,
    K: int,
    B: int,
    E: int,
    save_dir: str = ".",
) -> str:
    """
    Save the word embeddings (Ew) to a checkpoint file.

    File name format: model_dim-<d>_radius-<R>_ratio-<K>-batch-<B>-epoch-<E>.ckpt

    Returns
    -------
    path : path to the saved file
    """
    import os

    filename = f"model_dim-{d}_radius-{R}_ratio-{K}-batch-{B}-epoch-{E}.ckpt"
    path = os.path.join(save_dir, filename)
    # Save only the word embeddings so they can be loaded directly into a classifier
    torch.save(model.Ew.weight.data.cpu(), path)
    print(f"Embeddings saved to: {path}")
    return path


checkpoint_path = save_model(model, d=d, R=R, K=K, B=B, E=E)

Embeddings saved to: ./model_dim-64_radius-5_ratio-5-batch-512-epoch-5.ckpt


## 3.2 — Step 5: Bigger training run (optional)

If compute allows, we launch a larger experiment with a bigger embedding dimension
and more epochs. The results will be compared in the few-shots classification notebook.


In [19]:
# Larger hyperparameters for a better model
d_large = 128
E_large = 10
B_large = 512
R_large = 5
K_large = 5

# Re-build loaders with possibly different R/K (same here, but shown for clarity)
collate_fn_large = make_collate_fn(K=K_large, vocab_size=VOCAB_SIZE)
train_loader_large = DataLoader(
    train_set, batch_size=B_large, shuffle=True, collate_fn=collate_fn_large, drop_last=False
)
valid_loader_large = DataLoader(
    valid_set, batch_size=B_large, shuffle=False, collate_fn=collate_fn_large, drop_last=False
)

model_large = Word2Vec(vocab_size=VOCAB_SIZE, d=d_large).to(device)
optimiser_large = torch.optim.Adam(model_large.parameters(), lr=1e-3)

for epoch in range(1, E_large + 1):
    train_loss = train_epoch(model_large, train_loader_large, optimiser_large, device)
    valid_loss, valid_acc = evaluate(model_large, valid_loader_large, device)
    print(
        f"[Large] Epoch {epoch}/{E_large}  "
        f"train_loss={train_loss:.4f}  "
        f"valid_loss={valid_loss:.4f}  "
        f"valid_acc={valid_acc:.4f}"
    )

save_model(model_large, d=d_large, R=R_large, K=K_large, B=B_large, E=E_large)
# Use './model_dim-128_radius-5_ratio-5-batch-512-epoch-10.ckpt'

train:   0%|          | 0/3074 [00:00<?, ?it/s]

eval:   0%|          | 0/762 [00:00<?, ?it/s]

[Large] Epoch 1/10  train_loss=0.5425  valid_loss=0.4852  valid_acc=0.9104


train:   0%|          | 0/3074 [00:00<?, ?it/s]

eval:   0%|          | 0/762 [00:00<?, ?it/s]

...

[Large] Epoch 9/10  train_loss=0.3797  valid_loss=0.5163  valid_acc=0.9115


train:   0%|          | 0/3074 [00:00<?, ?it/s]

eval:   0%|          | 0/762 [00:00<?, ?it/s]

[Large] Epoch 10/10  train_loss=0.3740  valid_loss=0.5245  valid_acc=0.9111
Embeddings saved to: ./model_dim-128_radius-5_ratio-5-batch-512-epoch-10.ckpt


'./model_dim-128_radius-5_ratio-5-batch-512-epoch-10.ckpt'